# 03 — Baseline CNN

On va construire et entraîner un **premier réseau de neurones convolutif (CNN)** from scratch.

Ce modèle est volontairement **simple** — son rôle est d'établir une **baseline** :
un score de référence qu'on cherchera à dépasser dans le notebook 04 avec ResNet.

**Ce qu'on va faire :**
1. Comprendre l'architecture d'un CNN
2. Construire le modèle
3. Choisir la fonction de loss et l'optimiseur
4. Entraîner le modèle
5. Visualiser les courbes d'apprentissage
6. Évaluer sur le test set

## 1. Imports et configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Chemins
DATA_PATH    = Path('../data/raw/chest_xray')
RESULTS_PATH = Path('../results')
RESULTS_PATH.mkdir(exist_ok=True)
(RESULTS_PATH / 'figures').mkdir(exist_ok=True)
(RESULTS_PATH / 'models').mkdir(exist_ok=True)

# Choix du device (processeur)
# MPS = puce Apple Silicon (M1/M2/M3), sinon CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device utilisé : {device}')
print(f'PyTorch : {torch.__version__}')

## 2. Préparer les données

On recopie les transformations et DataLoaders du notebook 02.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
ROTATION_FILL = 128
BATCH_SIZE    = 32

train_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10, fill=ROTATION_FILL),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

train_dataset = datasets.ImageFolder(DATA_PATH / 'train', transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATA_PATH / 'val',   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(DATA_PATH / 'test',  transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Class weights depuis la config sauvegardée
with open(RESULTS_PATH / 'preprocessing_config.json') as f:
    config = json.load(f)

class_weights = torch.tensor(
    [config['weight_normal'], config['weight_pneumonia']],
    dtype=torch.float
).to(device)

print(f'Train : {len(train_dataset)} images | Val : {len(val_dataset)} | Test : {len(test_dataset)}')
print(f'Class weights → NORMAL: {class_weights[0]:.3f} | PNEUMONIA: {class_weights[1]:.3f}')

## 3. Architecture du CNN

Un CNN enchaîne deux types de blocs :

**Bloc convolutif** — extrait des caractéristiques visuelles (contours, textures, opacités...)
```
Conv2d → BatchNorm → ReLU → MaxPool
```
- **Conv2d** : applique des filtres sur l'image pour détecter des patterns
- **BatchNorm** : normalise les valeurs pour stabiliser l'entraînement
- **ReLU** : activation non-linéaire (met les valeurs négatives à 0)
- **MaxPool** : réduit la taille de moitié en gardant les valeurs max

**Bloc fully-connected (FC)** — prend les features extraites et fait la classification
```
Flatten → Linear → ReLU → Dropout → Linear → sortie
```
- **Flatten** : transforme la carte 2D en vecteur 1D
- **Dropout** : désactive aléatoirement des neurones → évite le surapprentissage
- **Linear** : couche de classification finale (2 sorties : NORMAL / PNEUMONIA)

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(BaselineCNN, self).__init__()

        # Bloc 1 : détecte les contours simples
        # Entrée : [batch, 3, 224, 224]
        # Sortie : [batch, 32, 112, 112]  (MaxPool divise par 2)
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # Bloc 2 : détecte des textures plus complexes
        # Sortie : [batch, 64, 56, 56]
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # Bloc 3 : détecte des patterns encore plus abstraits
        # Sortie : [batch, 128, 28, 28]
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # Bloc 4 : features de haut niveau
        # Sortie : [batch, 256, 14, 14]
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        # Classificateur final
        # Flatten : 256 × 14 × 14 = 50176 valeurs
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(p=0.5),   # Désactive 50% des neurones pendant l'entraînement
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x

model = BaselineCNN(num_classes=2).to(device)

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables : {total_params:,}')
print(model)

## 4. Fonction de loss et optimiseur

**Loss (CrossEntropyLoss) :** mesure à quel point la prédiction du modèle est mauvaise.
Plus la loss est faible, mieux le modèle prédit. On lui passe les `class_weights`
pour pénaliser davantage les erreurs sur la classe minoritaire (NORMAL).

**Optimiseur (Adam) :** algorithme qui ajuste les poids du modèle pour réduire la loss.
- `lr=0.001` : learning rate — la taille des pas d'ajustement.
  Trop grand → le modèle oscille. Trop petit → l'apprentissage est très lent.

**Scheduler (ReduceLROnPlateau) :** réduit automatiquement le learning rate
si la validation ne s'améliore plus pendant 3 époques. Le modèle peut alors
affiner ses poids avec des pas plus petits.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

print('Loss     : CrossEntropyLoss avec class weights')
print('Optimiseur : Adam (lr=0.001)')
print('Scheduler  : ReduceLROnPlateau (patience=3, factor=0.5)')

## 5. Boucle d'entraînement

Une **époque** = le modèle voit une fois toutes les images du train.

À chaque batch, la boucle fait :
1. **Forward pass** : le modèle prédit les classes
2. **Calcul de la loss** : on mesure l'erreur
3. **Backward pass** : on calcule les gradients (comment modifier les poids)
4. **Mise à jour** : l'optimiseur ajuste les poids

⚠️ **Sur Mac CPU, l'entraînement prend environ 3-5 min par époque.**
On entraîne 10 époques. Laisse tourner sans fermer le notebook.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # Mode entraînement (active Dropout et BatchNorm)
    total_loss, correct, total = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()          # Remet les gradients à zéro
        outputs = model(images)        # Forward pass
        loss = criterion(outputs, labels)  # Calcul de la loss
        loss.backward()                # Backward pass
        optimizer.step()               # Mise à jour des poids

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 20 == 0:
            print(f'  Batch {batch_idx+1}/{len(loader)} — loss: {loss.item():.4f}', end='\r')

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()  # Mode évaluation (désactive Dropout)
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():  # Pas de calcul de gradients → plus rapide
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

print('Fonctions train/evaluate définies ✓')

In [ ]:
NUM_EPOCHS = 10

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

print(f'Entraînement sur {NUM_EPOCHS} époques — device: {device}')
print('=' * 65)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Sauvegarde du meilleur modèle
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULTS_PATH / 'models' / 'baseline_cnn_best.pt')
        saved = '✓ sauvegardé'
    else:
        saved = ''

    print(f'Époque {epoch+1:>2}/{NUM_EPOCHS} | '
          f'Train loss: {train_loss:.4f} acc: {train_acc:.3f} | '
          f'Val loss: {val_loss:.4f} acc: {val_acc:.3f} {saved}')

print('=' * 65)
print('Entraînement terminé !')

## 6. Courbes d'apprentissage

Les courbes de loss et d'accuracy permettent de diagnostiquer l'entraînement :

- **Train loss descend, val loss aussi** → le modèle apprend bien
- **Train loss descend mais val loss remonte** → surapprentissage (overfitting)
- **Les deux stagnent** → le modèle n'apprend plus (learning rate trop faible ou modèle trop simple)

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, history['val_loss'],   'r-o', label='Validation')
axes[0].set_title('Courbe de Loss', fontsize=13)
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs, history['val_acc'],   'r-o', label='Validation')
axes[1].set_title("Courbe d'Accuracy", fontsize=13)
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Baseline CNN — Courbes d\'apprentissage', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_PATH / 'figures' / 'baseline_cnn_curves.png', dpi=150)
plt.show()

## 7. Évaluation finale sur le test set

On charge le meilleur modèle sauvegardé et on l'évalue sur les données de test
(données que le modèle n'a **jamais vues** pendant l'entraînement).

In [ ]:
# Charger le meilleur modèle
model.load_state_dict(torch.load(RESULTS_PATH / 'models' / 'baseline_cnn_best.pt', map_location=device))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print('=== Rapport de classification ===')
print(classification_report(
    all_labels, all_preds,
    target_names=['NORMAL', 'PNEUMONIA']
))

**Lecture du rapport :**
- **Precision** : parmi les cas prédits PNEUMONIA, combien sont vraiment PNEUMONIA ?
- **Recall** : parmi les vrais cas PNEUMONIA, combien a-t-on détectés ?
- **F1-score** : moyenne harmonique de precision et recall (le plus important ici)

> En médecine, le **Recall PNEUMONIA** est crucial : rater un cas de pneumonie
> (faux négatif) est bien plus grave qu'une fausse alarme (faux positif).

In [ ]:
# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['NORMAL', 'PNEUMONIA'],
    yticklabels=['NORMAL', 'PNEUMONIA'],
    ax=ax
)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Réalité', fontsize=12)
ax.set_title('Matrice de confusion — Baseline CNN', fontsize=13)

# Annoter les quadrants
ax.text(0.5, -0.15, 'Vrais Négatifs (TN)  |  Faux Positifs (FP)\nFaux Négatifs (FN)  |  Vrais Positifs (TP)',
        transform=ax.transAxes, ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig(RESULTS_PATH / 'figures' / 'baseline_cnn_confusion.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Vrais Positifs  (TP) : {tp}  — pneumonie détectée correctement')
print(f'Vrais Négatifs  (TN) : {tn}  — normal détecté correctement')
print(f'Faux Positifs   (FP) : {fp}  — normal prédit comme pneumonie (fausse alarme)')
print(f'Faux Négatifs   (FN) : {fn}  — pneumonie manquée (dangereux !)')

## 8. Conclusions

Note ici tes observations :

- Quelle accuracy obtiens-tu sur le test set ?
- Le Recall PNEUMONIA est-il satisfaisant ?
- Voit-on de l'overfitting sur les courbes ?
- Combien de cas de pneumonie ont été manqués (FN) ?

---
**Prochain notebook :** `04_transfer_learning.ipynb`
On remplace ce CNN simple par **ResNet18** pré-entraîné sur ImageNet.
Objectif : dépasser ce score de baseline avec beaucoup moins d'époques.